**Fine-Tuning a Transformer (BERT) for Extractive Question Answering on SQuAD v1.1**

This notebook fine-tunes `bert-base-uncased` on the SQuAD v1.1 dataset using the Hugging Face `transformers`, `datasets`, and `evaluate` libraries, evaluates it with Exact Match (EM) and F1, and tests it on custom question/context pairs.


## Introduction

### Classification vs. Question Answering

**Text classification** tasks map an entire input (a sentence, paragraph, or document) to a single label from a fixed, predefined set of classes. For example, sentiment analysis maps a review to `{positive, negative, neutral}`. The model outputs one probability distribution over a small, closed label space, and the loss is typically cross-entropy over that space.

**Extractive Question Answering (QA)**, as done in SQuAD, is fundamentally different:

- The "labels" are not fixed categories -- they are **positions in the input text** (a start token index and an end token index) that mark the span containing the answer.
- The model receives a **pair** of sequences (question + context), not a single sequence, and must reason about how the question relates to specific tokens inside the context.
- The output space changes with every example, since it depends on the length and content of that example's context -- there is no fixed answer vocabulary.
- The model architecture reflects this: instead of one classification head, a QA model has **two** linear heads on top of BERT's token representations, one predicting the probability that each token is the answer's start, and one predicting the probability that each token is the answer's end. The final answer is the highest-scoring valid `(start, end)` span.
- Evaluation is also different: instead of accuracy over classes, QA is evaluated with **Exact Match (EM)** -- did the predicted span exactly match a reference answer (after normalization) -- and **F1** -- token-level overlap between predicted and reference answer, which gives partial credit for close-but-imperfect spans.

In short: classification predicts *what class the whole input belongs to*; extractive QA predicts *where in the input the answer lives*.


## Environment Setup

In [ ]:
!pip install -q transformers datasets evaluate accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch
import collections
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
import evaluate

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Torch version: 2.11.0+cu128
CUDA available: True
Using device: cuda


## Load and Explore the Dataset


In [ ]:
dataset = load_dataset("rajpurkar/squad")
dataset


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [ ]:
# Inspect a single example
dataset["train"][0]


{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

In [ ]:
# Look at a few more examples to understand structure
for i in range(3):
    ex = dataset["train"][i]
    print("Question:", ex["question"])
    print("Context (truncated):", ex["context"][:200], "...")
    print("Answer:", ex["answers"])
    print("-" * 80)


Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Context (truncated): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta ...
Answer: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}
--------------------------------------------------------------------------------
Question: What is in front of the Notre Dame Main Building?
Context (truncated): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta ...
Answer: {'text': ['a copper statue of Christ'], 'answer_start': [188]}
--------------------------------------------------------------------------------
Question: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
Context (trunc

## Tokenization and Preprocessing


In [ ]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

max_length = 384      # max length of a feature (question + context)
doc_stride = 128      # overlap between chunks when context is split


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def prepare_train_features(examples):
    examples["question"] = [q.lstrip() for q in examples["question"]]

    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized_examples.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:

            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)

                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples


In [ ]:
def prepare_validation_features(examples):
    examples["question"] = [q.lstrip() for q in examples["question"]]

    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")

    tokenized_examples["example_id"] = []

    for i in range(len(tokenized_examples["input_ids"])):
        sequence_ids = tokenized_examples.sequence_ids(i)
        context_index = 1

        sample_index = sample_mapping[i]
        tokenized_examples["example_id"].append(examples["id"][sample_index])

        tokenized_examples["offset_mapping"][i] = [
            (o if sequence_ids[k] == context_index else None)
            for k, o in enumerate(tokenized_examples["offset_mapping"][i])
        ]

    return tokenized_examples


In [ ]:
train_subset = dataset["train"].select(range(8000))
val_subset = dataset["validation"].select(range(1000))

tokenized_train = train_subset.map(
    prepare_train_features,
    batched=True,
    remove_columns=train_subset.column_names,
)

tokenized_val = val_subset.map(
    prepare_validation_features,
    batched=True,
    remove_columns=val_subset.column_names,
)

print(tokenized_train)
print(tokenized_val)


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 8099
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 1020
})


## Model Setup

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
model.to(device)
print(model.config.architectures)


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

['BertForMaskedLM']


## Fine-Tuning the Model


In [ ]:
args = TrainingArguments(
    output_dir="bert-squad-finetuned",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_eval_columns = tokenized_val.remove_columns(["example_id", "offset_mapping"])

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=trainer_eval_columns,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)


In [ ]:
train_result = trainer.train()
train_result


Epoch,Training Loss,Validation Loss
1,0.718038,No log
2,0.667541,No log
3,0.395681,No log


TrainOutput(global_step=1521, training_loss=0.5917600535155126, metrics={'train_runtime': 464.3329, 'train_samples_per_second': 52.327, 'train_steps_per_second': 3.276, 'total_flos': 4761545473810944.0, 'train_loss': 0.5917600535155126, 'epoch': 3.0})

In [ ]:
trainer.save_model("bert-squad-finetuned")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Evaluation

The Trainer's built-in `evaluate()` only reports loss (QA models don't have a simple per-token accuracy metric). To get **Exact Match** and **F1**, we need to:
1. Run the model on validation features to get start/end logits.
2. Convert logits back into predicted text spans (post-processing), picking the best-scoring valid span per example.
3. Compare predicted spans against the reference answers using the official SQuAD metric via `evaluate`.


In [ ]:
raw_predictions = trainer.predict(trainer_eval_columns)


In [ ]:
def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = raw_predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature_example_id in enumerate(features["example_id"]):
        features_per_example[example_id_to_index[feature_example_id]].append(i)

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        context = example["context"]

        valid_answers = []
        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logits[start_index] + end_logits[end_index],
                            "text": context[start_char:end_char],
                        }
                    )

        if len(valid_answers) > 0:
            best_answer = max(valid_answers, key=lambda x: x["score"])
        else:
            best_answer = {"text": "", "score": 0.0}

        predictions[example["id"]] = best_answer["text"]

    return predictions


In [ ]:
final_predictions = postprocess_qa_predictions(val_subset, tokenized_val, raw_predictions.predictions)

squad_metric = evaluate.load("squad")

formatted_predictions = [{"id": k, "prediction_text": v} for k, v in final_predictions.items()]
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in val_subset]

results = squad_metric.compute(predictions=formatted_predictions, references=references)
print("Validation results:")
print(f"  Exact Match: {results['exact_match']:.2f}")
print(f"  F1 Score:    {results['f1']:.2f}")


Validation results:
  Exact Match: 73.30
  F1 Score:    79.88


### Results


- **Validation Exact Match (EM):** `73.30`
- **Validation F1 Score:** `79.88`

## Testing on Custom Examples

We load the fine-tuned model in a simple `question, context -> answer` inference function and try it on several custom examples.


In [ ]:
def answer_question(question, context, model=model, tokenizer=tokenizer):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    start_index = torch.argmax(outputs.start_logits)
    end_index = torch.argmax(outputs.end_logits)

    input_ids = inputs["input_ids"][0]
    answer_tokens = input_ids[start_index: end_index + 1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    return answer


In [ ]:
# Custom test 1
question_1 = "Who developed the theory of relativity?"
context_1 = "Albert Einstein developed the theory of relativity in the early 20th century."
print("Q:", question_1)
print("A:", answer_question(question_1, context_1))


Q: Who developed the theory of relativity?
A: albert einstein


In [ ]:
# Custom test 2
question_2 = "What is the capital of France?"
context_2 = "France is a country in Western Europe. Its capital, Paris, is known for the Eiffel Tower and the Louvre Museum."
print("Q:", question_2)
print("A:", answer_question(question_2, context_2))


Q: What is the capital of France?
A: paris


In [ ]:
# Custom test 3
question_3 = "When was the Hugging Face Transformers library released?"
context_3 = (
    "Hugging Face is a company that develops tools for building applications using machine learning. "
    "Its Transformers library, first released in 2018, provides thousands of pretrained models "
    "for tasks such as text classification, question answering, and translation."
)
print("Q:", question_3)
print("A:", answer_question(question_3, context_3))


Q: When was the Hugging Face Transformers library released?
A: 2018


## Summary and Takeaways

Fine-tuning BERT for extractive QA highlights how span-prediction differs from ordinary sequence classification: instead of predicting a single categorical label per example, the model scores every token position as a potential start or end index of an answer span.

Key technical highlights of this pipeline:
- **Tokenization with Sliding Windows:** Handled document contexts exceeding BERT's maximum sequence length (384 tokens) using `stride=128` and `return_overflowing_tokens=True`.
- **Offset Mapping:** Converted character-level gold answer spans in raw context text into token-level target start/end indices, mapping out-of-span answers to the `[CLS]` token.
- **Post-Processing & Span Scoring:** Logits from overlapping feature windows were aggregated, invalid spans filtered (e.g. end < start, span length > 30), and top candidate spans scored to select best text predictions.
- **Evaluation:** Evaluated on SQuAD v1.1 validation split using Exact Match (EM) and token-level F1 metrics.

Overall, this implementation demonstrates an end-to-end extractive question answering pipeline using the Hugging Face ecosystem (`transformers`, `datasets`, and `evaluate`), from raw text preprocessing to token-level span prediction and post-processing evaluation.